# MinIO + Apache Spark Analytics Example

This notebook demonstrates how to use MinIO as a data lake with Apache Spark for analytics.

## 1. Setup Spark Session with MinIO

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os

# Create Spark session with MinIO configuration
spark = SparkSession.builder \
    .appName("MinIO Analytics") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "myminio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 2. Read Application Logs from MinIO

In [ ]:
# Read JSON logs from app-logs bucket
logs_df = spark.read.json("s3a://app-logs/*/*.json")

# Show schema
logs_df.printSchema()

# Display sample data
logs_df.show(10, truncate=False)

## 3. Log Analytics - Aggregations

In [ ]:
# Count logs by level
log_level_counts = logs_df.groupBy("level").count().orderBy(desc("count"))
log_level_counts.show()

# Logs by service
service_counts = logs_df.groupBy("service").count().orderBy(desc("count"))
service_counts.show()

In [ ]:
# Time-based analysis - logs per hour
logs_with_hour = logs_df.withColumn(
    "hour", 
    hour(to_timestamp(col("timestamp")))
)

hourly_logs = logs_with_hour.groupBy("hour", "level") \
    .count() \
    .orderBy("hour", "level")

hourly_logs.show(24)

## 4. Error Pattern Detection

In [ ]:
# Filter error logs and extract patterns
error_logs = logs_df.filter(col("level") == "ERROR")

# Word frequency in error messages
error_words = error_logs.select(
    explode(split(lower(col("message")), " ")).alias("word")
).filter(
    length(col("word")) > 3  # Filter short words
).groupBy("word").count().orderBy(desc("count"))

print("Most common words in error messages:")
error_words.show(20)

## 5. Save Analytics Results to MinIO (Parquet)

In [ ]:
# Save aggregated results as Parquet for efficient querying
daily_summary = logs_df.withColumn(
    "date", to_date(col("timestamp"))
).groupBy("date", "level", "service").agg(
    count("*").alias("log_count"),
    collect_list("message").alias("messages")
)

# Write to analytics bucket
daily_summary.write \
    .mode("overwrite") \
    .partitionBy("date") \
    .parquet("s3a://analytics-data/daily_log_summary")

print("Analytics results saved to s3a://analytics-data/daily_log_summary")

## 6. Create Hive External Table

In [ ]:
# Enable Hive support
spark.sql("CREATE DATABASE IF NOT EXISTS analytics")
spark.sql("USE analytics")

# Create external table pointing to MinIO
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS log_events (
        id STRING,
        timestamp TIMESTAMP,
        level STRING,
        message STRING,
        service STRING
    )
    STORED AS PARQUET
    LOCATION 's3a://analytics-data/log_events'
""")

# Query using SQL
spark.sql("""
    SELECT level, COUNT(*) as count 
    FROM log_events 
    GROUP BY level
""").show()

## 7. Streaming Analytics (Real-time)

In [ ]:
# Structured Streaming from MinIO
# Monitor new logs as they arrive

schema = StructType([
    StructField("id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("level", StringType(), True),
    StructField("message", StringType(), True),
    StructField("service", StringType(), True)
])

# Stream from MinIO bucket
streaming_df = spark.readStream \
    .schema(schema) \
    .json("s3a://app-logs/")

# Real-time aggregation
real_time_counts = streaming_df \
    .withWatermark("timestamp", "1 minute") \
    .groupBy(
        window(col("timestamp"), "5 minutes"),
        col("level")
    ).count()

# Output to console (for demo)
query = real_time_counts.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

# Wait for 60 seconds then stop
import time
time.sleep(60)
query.stop()

## 8. Cleanup

In [ ]:
spark.stop()